## Analisis inicial y Union
Primera parte del exploratorio, lo que quiero ver es como unir los 3 excels juntos y cuan mal están:

Los leo:

In [83]:
from datetime import datetime, time, timedelta
from pathlib import Path

import pandas as pd

# Lo pongo así para que no se rompa el path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

EXCEL_DIR = ROOT / "data" / "excels"
archivos = sorted(EXCEL_DIR.glob("Totalizadores Planta *.xlsx"))

# Por ahora miro solo EE para no tener que ver muchas hojas a la vez:
HOJA_REF = "Consolidado EE"
LIMITE = 23 * 3600            # 23:00:00 en segundos desde medianoche

# Se leen de nuevo en cada ejecución
dfs = {}

for archivo in archivos:
    print("Leyendo", archivo.name)
    dfs[archivo.name] = pd.read_excel(archivo, sheet_name=HOJA_REF, engine="openpyxl")

for nombre, df in dfs.items():
    print(nombre, df.shape, list(df.columns[:3]))

Leyendo Totalizadores Planta 2020_2022.xlsx
Leyendo Totalizadores Planta 2021_2023.xlsx
Leyendo Totalizadores Planta 2022_2023.xlsx
Totalizadores Planta 2020_2022.xlsx (15575, 21) ['DIA', 'HORA', 'Planta (Kw)']
Totalizadores Planta 2021_2023.xlsx (15316, 21) ['DIA', 'HORA', 'Planta (Kw)']
Totalizadores Planta 2022_2023.xlsx (12009, 21) ['DIA', 'HORA', 'Planta (Kw)']


In [84]:
def a_segundos(v):
    """Convierte una celda de hora a segundos desde las 00:00, o None si no se entiende."""
    if v is None or (not isinstance(v, str) and pd.isna(v)):
        return None
    if isinstance(v, datetime):
        v = v.time()
    if isinstance(v, time):
        return v.hour * 3600 + v.minute * 60 + v.second
    if isinstance(v, timedelta):
        return int(v.total_seconds())
    if isinstance(v, (int, float)):
        return round(v * 86400)
    if isinstance(v, str):
        partes = v.strip().split(":")
        if len(partes) == 3 and all(p.isdigit() for p in partes):
            h, m, s = map(int, partes)
            return h * 3600 + m * 60 + s
    return None                               # formato raro, como "0:1:00:01"


def formato(seg):
    return f"{seg // 3600:02d}:{seg % 3600 // 60:02d}:{seg % 60:02d}"


def revisar_horas(df):
    """Horas posteriores a las 23:00:00 y valores de hora con formato raro."""
    horas = df.iloc[:, 1]                 # la segunda columna es la hora
    segundos = horas.map(a_segundos)

    tarde = segundos[segundos > LIMITE]
    print("Horas > 23:00:00 (valor: cantidad de filas)")
    for seg, n in tarde.value_counts().sort_index().items():
        print(f"  {formato(int(seg))}: {n}")

    raros = horas[segundos.isna() & horas.notna()]
    if len(raros):
        print("Valores con formato no reconocido:")
        for valor, n in raros.astype(str).value_counts().items():
            print(f"  {valor!r}: {n}")


def revisar_dias(df, min_filas=24, max_filas=25):
    """Filas por día, y un df aparte con los días con menos de min_filas o más de max_filas."""
    col_dia = df.columns[0]
    conteo = df.groupby(col_dia).size()

    print("Cantidad de días según filas por día:")
    print(conteo.value_counts().sort_index())

    dias_raros = conteo[(conteo < min_filas) | (conteo > max_filas)].index
    df_raros = df[df[col_dia].isin(dias_raros)].copy()
    return conteo, df_raros

Me ponga a ver como es el formato de las :59 

**queda justificar por que las :59**

In [85]:
for nombre, df in dfs.items():
    print(f"\n=== {nombre} ===")
    revisar_horas(df)


=== Totalizadores Planta 2020_2022.xlsx ===
Horas > 23:00:00 (valor: cantidad de filas)
  23:00:01: 10
  23:59:00: 614
  23:59:01: 7
  23:59:59: 1

=== Totalizadores Planta 2021_2023.xlsx ===
Horas > 23:00:00 (valor: cantidad de filas)
  23:00:01: 9
  23:59:00: 605
  23:59:01: 5
  23:59:59: 1

=== Totalizadores Planta 2022_2023.xlsx ===
Horas > 23:00:00 (valor: cantidad de filas)
  23:00:01: 5
  23:59:00: 475
  23:59:01: 3
  23:59:59: 1


La mayoría de horas están bien, hay que tener en cuenta las 23:00:01 seguramente son duplicados de 23:00 y no de :59. 

In [86]:
conteos = {}
lista_raros = []

for nombre, df in dfs.items():
    print(f"\n=== {nombre} ===")
    conteos[nombre], df_raros = revisar_dias(df)
    lista_raros.append(df_raros)

raros = pd.concat(lista_raros, ignore_index=True)
print(raros.shape)


=== Totalizadores Planta 2020_2022.xlsx ===
Cantidad de días según filas por día:
8        1
23       2
24       5
25     596
50       5
51       1
100      2
Name: count, dtype: int64

=== Totalizadores Planta 2021_2023.xlsx ===
Cantidad de días según filas por día:
9        1
16       1
23       3
24       3
25     586
50       5
51       1
99       1
100      1
Name: count, dtype: int64

=== Totalizadores Planta 2022_2023.xlsx ===
Cantidad de días según filas por día:
8       1
16      1
17      1
20      1
23      1
25    463
50      4
51      1
99      1
Name: count, dtype: int64
(1583, 21)


Hay como 100 días pegados. Saco ahora todos los valores que no sean 
- 23:59:00
- 23:59:01
- 23:59:59

In [87]:
def filtrar_2359(df):
    """Deja solo las filas con hora 23:59:xx (23:59:00, 23:59:01, 23:59:59)."""
    seg = pd.to_numeric(df.iloc[:, 1].map(a_segundos), errors="coerce")
    inicio = 23 * 3600 + 59 * 60          # 23:59:00
    fin = inicio + 59                     # 23:59:59
    return df[(seg >= inicio) & (seg <= fin)].copy()

In [88]:
dfs_2359 = {}
col_dia = "DIA"

for nombre, df in dfs.items():
    f = filtrar_2359(df)
    dfs_2359[nombre] = f

# Veo para una tabla:
f.info()

<class 'pandas.DataFrame'>
Index: 479 entries, 24 to 12000
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   DIA                   479 non-null    datetime64[us]
 1   HORA                  479 non-null    str           
 2   Planta (Kw)           479 non-null    float64       
 3   Elaboracion (Kw)      479 non-null    int64         
 4   Bodega (Kw)           479 non-null    int64         
 5   Cocina (Kw)           479 non-null    int64         
 6   Envasado (Kw)         479 non-null    int64         
 7   Linea 2 (Kw)          479 non-null    float64       
 8   Linea 3 (Kw)          479 non-null    int64         
 9   Linea 4 (Kw)          479 non-null    int64         
 10  Servicios (Kw)        479 non-null    int64         
 11  Sala Maq (Kw)         479 non-null    int64         
 12  Aire (Kw)             479 non-null    int64         
 13  Calderas (Kw)         479 non-nul

In [91]:
# Hago el dict con todo el df: 
# dfs_todo[archivo][hoja]
dfs_todo = {}

for archivo in archivos:
    print("Leyendo", archivo.name)
    dfs_todo[archivo.name] = pd.read_excel(archivo, sheet_name=None, engine="openpyxl")
    # sheet_name=None lee todas las hojas y devuelve un dict {nombre_hoja: df}

# Faltan hojas a un archivo
hojas_por_archivo = {n: set(h.keys()) for n, h in dfs_todo.items()}
todas = set.union(*hojas_por_archivo.values())
for n, h in hojas_por_archivo.items():
    print(n, len(h), "hojas | faltan:", todas - h)

Leyendo Totalizadores Planta 2020_2022.xlsx
Leyendo Totalizadores Planta 2021_2023.xlsx
Leyendo Totalizadores Planta 2022_2023.xlsx
Totalizadores Planta 2020_2022.xlsx 15 hojas | faltan: {'Seguimiento Dia'}
Totalizadores Planta 2021_2023.xlsx 16 hojas | faltan: set()
Totalizadores Planta 2022_2023.xlsx 15 hojas | faltan: {'Seguimiento Dia'}


Borro seguimiento día por que está en solo un excel

In [93]:
dfs_todo["Totalizadores Planta 2021_2023.xlsx"].pop("Seguimiento Dia")

,DIA,HORA,Ultimo Dato del Dia,Unnamed: 3,Unnamed: 4
0,2021-07-01,02:00:00,0,20002,NaN
1,2021-07-01,03:00:00,0,20003,NaN
2,2021-07-01,04:00:00,0,20004,NaN
3,2021-07-01,05:00:01,0,20005,NaN
4,2021-07-01,06:00:01,0,20006,NaN
...,...,...,...,...,...
15311,2023-03-06,04:00:00,0,35313,NaN
15312,2023-03-06,05:00:00,0,35314,NaN
15313,2023-03-06,06:00:00,0,35315,NaN
15314,2023-03-06,07:00:00,0,35316,NaN


### Union de los datos

In [94]:
unidos = {}
todas_las_hojas = sorted({h for hojas in dfs_todo.values() for h in hojas})

for hoja in todas_las_hojas:
    partes = []
    for nombre, hojas in dfs_todo.items():
        partes.append(filtrar_2359(hojas[hoja]))

    df = pd.concat(partes, ignore_index=True)
    df["DIA"] = pd.to_datetime(df["DIA"], errors="coerce").dt.date
    unidos[hoja] = df

    print(f"{hoja:35s} {df.shape} | DIA no interpretable: {df['DIA'].isna().sum()}")

Auxiliar                            (1233, 37) | DIA no interpretable: 0
Consolidado Agua                    (1712, 24) | DIA no interpretable: 0
Consolidado Aire                    (1712, 14) | DIA no interpretable: 0
Consolidado EE                      (1712, 21) | DIA no interpretable: 0
Consolidado GasVapor                (1712, 21) | DIA no interpretable: 0
Consolidado KPI                     (1712, 126) | DIA no interpretable: 0
Consolidado Produccion              (1712, 18) | DIA no interpretable: 0
Totalizadores Agua                  (1712, 44) | DIA no interpretable: 0
Totalizadores Aire                  (1712, 12) | DIA no interpretable: 0
Totalizadores CO2                   (1713, 9) | DIA no interpretable: 0
Totalizadores Efluentes             (1712, 9) | DIA no interpretable: 0
Totalizadores Energia               (1712, 60) | DIA no interpretable: 0
Totalizadores Gas y Vapor           (1712, 25) | DIA no interpretable: 0
Totalizadores Glicol                (1711, 12) | DIA

In [95]:
with pd.ExcelWriter(
    ROOT / "data" / "processed" / "unido_2359.xlsx",
    engine="openpyxl",
    date_format="YYYY/MM/DD", # Si no lo escribo así pone otra vez el 00:00:00
) as writer:
    for hoja, df in unidos.items():
        df.to_excel(writer, sheet_name=hoja[:31], index=False)

## Analisis del DF completo